# PHINC - Difficulty Assignment (Hinglish to English translation)

Assigns `difficulty` (Easy / Medium / Hard) to 200 sampled PHINC rows.

**Task.** `question` is a code-mixed Hinglish social-media message, `answer` is
its English translation. The models must generate a translation - no options.

**Two-pass design.** A translation score is continuous, so there is no natural
right/wrong. This notebook therefore runs in two passes:

1. **Score** every row with every model and store the raw chrF++ value.
2. **Derive a threshold** from those scores (Cell 9), then a model "passes" a
   row when it clears the threshold. The three pass/fail votes sum as usual:

| Models passing | Difficulty |
|---|---|
| 3 / 3 | Easy |
| 2 / 3 | Medium |
| 0-1 / 3 | Hard |

Because raw scores are stored, **re-deriving difficulty at a different
threshold costs nothing** - no model is re-run.

**Metric: chrF++, not ROUGE-L.** ROUGE-L is a summarisation metric; chrF++ is
the standard for Indic MT, deterministic, needs no extra model, and handles the
informal English references here better than BLEU. Measured on this data, a
system that simply **copies the Hinglish input** without translating scores
~44% chrF - so Cell 4 computes that floor explicitly and Cell 9 uses it as a
sanity marker when picking the threshold.

**Output.** `phinc_difficulty.jsonl` - the original 14 schema fields with
`difficulty` filled in, plus an audit file holding every model's translation
and score.

### Cell 1 - Install dependencies and authenticate

Adds `sacrebleu` to the usual stack - it provides the reference implementation
of chrF++ with a reproducible signature, which is safer than hand-rolling
character n-gram scoring.

**An HF token is required.** Llama-3.1 and Gemma-2 are gated; only Mistral is
open. Accept each licence on huggingface.co, create a **read** token, then add
it in Colab via the **key icon** as a secret named `HF_TOKEN`. Use the secret
rather than pasting the token into a cell.

In [1]:
!pip -q install -U transformers accelerate bitsandbytes huggingface_hub sacrebleu

import sacrebleu
print("sacrebleu", sacrebleu.__version__)

HF_OK = False
try:
    from google.colab import userdata
    from huggingface_hub import login
    login(token=userdata.get("HF_TOKEN"))
    HF_OK = True
    print("HF login OK")
except Exception as e:
    print("No HF token ({}: {})".format(type(e).__name__, e))
    print("Mistral will still work; gated Llama/Gemma will fail with a 401.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 19.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 795.8/795.8 kB 39.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.6/129.6 kB 11.9 MB/s eta 0:00:00
sacrebleu 2.6.0
HF login OK


### Cell 2 - Mount Drive

Drive is the weight cache: models are downloaded once, quantised to 4-bit and
saved here, so later runs skip the download entirely.

`DRIVE_OK` records whether the mount actually succeeded - later cells check it
rather than assuming. If you see **"credential propagation was unsuccessful"**,
the auth popup did not complete: re-run and finish it, allow pop-ups and
third-party cookies for `colab.research.google.com`, or mount from the Files
sidebar. Without Drive everything still runs, just uncached.

In [2]:
import os

DRIVE_OK    = False
DRIVE_MOUNT = "/drive"
CACHE_DIR   = os.path.join(DRIVE_MOUNT, "MyDrive", "models")

try:
    from google.colab import drive
    drive.mount(DRIVE_MOUNT, force_remount=True)
    DRIVE_OK = os.path.isdir(os.path.join(DRIVE_MOUNT, "MyDrive"))
except ImportError:
    print("Not running on Colab - Drive caching disabled.")
except Exception as e:
    print("DRIVE MOUNT FAILED: {}".format(e))

if DRIVE_OK:
    os.makedirs(CACHE_DIR, exist_ok=True)
    print("Drive mounted | weight cache: {}".format(CACHE_DIR))
else:
    print("\nWARNING: no Drive - weights will NOT be cached between sessions.")

Mounted at /drive
Drive mounted | weight cache: /drive/MyDrive/models


### Cell 3 - Configuration

- `THRESHOLD_MODE` - how Cell 9 picks the pass mark.
  - `"auto_median"` (default) - the median of all pooled model scores. Makes
    the vote a relative judgment and gives the most balanced Easy/Medium/Hard
    spread, which is what you want from a difficulty label.
  - `"baseline"` - the copy-the-input floor from Cell 4. Semantically stronger
    ("did the model beat not translating at all?") but lenient, so most rows
    land in Easy.
  - `"fixed"` - use `FIXED_THRESHOLD` verbatim.
- `SET_EVAL_METRIC` - leave `None` to keep whatever `eval_metric` the input
  file already carries (currently `chrF++`, which matches what this notebook
  scores with). Only set it if you want the output rows to record a different
  metric name.
- `MODELS` - three judges from three families (Mistral / Meta / Google) so
  their errors decorrelate. `USE_INSTRUCT` picks instruction-tuned checkpoints,
  which follow a translation instruction far better; Cell 5 detects base vs
  chat automatically either way.

In [3]:
import gc
import re
import json
import random
import shutil
import statistics
from collections import Counter

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

# ---- paths ----
INPUT_FILE  = "phinc.jsonl"
OUTPUT_FILE = "phinc_difficulty.jsonl"
AUDIT_FILE  = "phinc_audit.jsonl"
PROG_DIR    = "judge_progress"

# ---- sampling ----
N_ROWS        = 300
SEED          = 42
MIN_REF_WORDS = 3        # drop rows whose reference translation is junk

# ---- thresholding ----
THRESHOLD_MODE  = "auto_median"     # auto_median | baseline | fixed
FIXED_THRESHOLD = 0.50

# ---- generation ----
MAX_NEW_TOKENS = 256
BATCH_SIZE     = 25

# ---- schema ----
SET_EVAL_METRIC = None              # None keeps the file's value; e.g. "chrf++"

# ---- the three judges ----
USE_INSTRUCT = True

REPOS = {
    True: {
        "mistral": "mistralai/Mistral-7B-Instruct-v0.3",   # ungated
        "llama":   "meta-llama/Llama-3.1-8B-Instruct",     # GATED
        "gemma":   "google/gemma-2-9b-it",                 # GATED
    },
    False: {
        "mistral": "mistralai/Mistral-7B-v0.3",
        "llama":   "meta-llama/Llama-3.1-8B",
        "gemma":   "google/gemma-2-9b",
    },
}[USE_INSTRUCT]

MODELS = [
    {"name": "mistral", "repo": REPOS["mistral"]},
    {"name": "llama",   "repo": REPOS["llama"]},
    {"name": "gemma",   "repo": REPOS["gemma"], "attn": "eager"},
]

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,     # T4 has no bf16
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
)

SCHEMA_KEYS = [
    "id", "source", "category", "subcategory", "region", "language",
    "difficulty", "task_type", "question", "options", "answer",
    "explanation", "cultural_attr", "eval_metric",
]

os.makedirs(PROG_DIR, exist_ok=True)
print("Judges ({}):".format("instruct" if USE_INSTRUCT else "base"))
for m in MODELS:
    cached = DRIVE_OK and os.path.isfile(
        os.path.join(CACHE_DIR, m["name"] + "_4bit", "config.json"))
    print("  {:<8} {:<42} {}".format(
        m["name"], m["repo"], "cached in Drive" if cached else "will download"))
print("\ndevice:", "cuda" if torch.cuda.is_available() else "CPU (will be very slow)")

Judges (instruct):
  mistral  mistralai/Mistral-7B-Instruct-v0.3         cached in Drive
  llama    meta-llama/Llama-3.1-8B-Instruct           cached in Drive
  gemma    google/gemma-2-9b-it                       cached in Drive

device: cuda


### Cell 4 - Load, sample, and measure the reference floors

Samples 200 rows under a fixed seed, so the same rows return on every run -
which is what makes the resume logic safe across sessions.

**Junk references are dropped first.** About 1.6% of PHINC rows have a
reference that is an artifact rather than a translation - `)`, `$`, `#039` (a
half-decoded HTML entity). A model that translates those sources perfectly
still scores near zero, so the row would be labelled Hard because the reference
is broken, not because the sentence is hard. `MIN_REF_WORDS` filters them out.

Then it computes two **model-free reference points** that Cell 9 uses to judge
whether a threshold is sensible:

- **Copy-the-input floor** - score the untranslated Hinglish source against the
  English reference. This is what a system that does nothing scores, and it is
  not near zero: source and target share English words, names and hashtags. Any
  threshold at or below this floor is meaningless.
- **Unrelated-output floor** - score a different row's translation. This is the
  true "wrong answer" level.

A useful threshold sits well above the first, and both are printed so you can
see the usable range before spending GPU time.

In [4]:
import sacrebleu
_CHRF = sacrebleu.CHRF(word_order=2)          # word_order=2 makes this chrF++


def chrf_pp(hypothesis, reference):
    if not hypothesis or not hypothesis.strip():
        return 0.0
    return _CHRF.sentence_score(hypothesis, [reference]).score / 100.0


with open(INPUT_FILE, encoding="utf-8") as f:
    all_rows = [json.loads(line) for line in f]
print("Loaded {} rows from {}".format(len(all_rows), INPUT_FILE))

def usable(row):
    # ~1.6% of PHINC references are artifacts rather than translations: ")",
    # "$", "#039" (a half-decoded HTML entity). A model that translates those
    # sources perfectly still scores near zero, so the row would be labelled
    # Hard because the reference is broken, not because the sentence is hard.
    ref = str(row.get("answer") or "").strip()
    return len(ref.split()) >= MIN_REF_WORDS and re.search(r"[A-Za-z]", ref)


usable_rows = [r for r in all_rows if usable(r)]
print("usable rows: {}/{}  ({} dropped for junk references)".format(
    len(usable_rows), len(all_rows), len(all_rows) - len(usable_rows)))
assert len(usable_rows) >= N_ROWS, "not enough usable rows to sample from"

random.seed(SEED)
sample = random.sample(usable_rows, N_ROWS)

lens = sorted(len(r["question"].split()) for r in sample)
print("Sampled {} rows | source words: min {}, median {}, max {}".format(
    len(sample), lens[0], lens[len(lens) // 2], lens[-1]))

# ---- model-free floors ----
copy_scores = [chrf_pp(r["question"], r["answer"]) for r in sample]
rand_scores = [chrf_pp(sample[(i + 7) % len(sample)]["answer"], r["answer"])
               for i, r in enumerate(sample)]

COPY_FLOOR = statistics.mean(copy_scores)
RAND_FLOOR = statistics.mean(rand_scores)

print("\nReference floors (chrF++, no model involved):")
print("  copy the Hinglish input verbatim : {:.1%}   <- a system that does nothing".format(COPY_FLOOR))
print("  an unrelated row's translation   : {:.1%}   <- true wrong-answer level".format(RAND_FLOOR))
print("  a perfect translation            : 100.0%")
print("\nUsable range for a threshold: roughly {:.0%} to 100%".format(COPY_FLOOR))

print("\n--- example row ---")
print("  source   :", " ".join(sample[0]["question"].split())[:96])
print("  reference:", " ".join(str(sample[0]["answer"]).split())[:96])

Loaded 2000 rows from phinc.jsonl
usable rows: 1964/2000  (36 dropped for junk references)
Sampled 300 rows | source words: min 3, median 11, max 36

Reference floors (chrF++, no model involved):
  copy the Hinglish input verbatim : 36.7%   <- a system that does nothing
  an unrelated row's translation   : 10.1%   <- true wrong-answer level
  a perfect translation            : 100.0%

Usable range for a threshold: roughly 37% to 100%

--- example row ---
  source   : @someUSER so they should've ran the ball on 3rd &amp
  reference: 7 ? blame cam all u want but flacco is the problem . elite qbs dont have games like that


### Cell 5 - Build the translation prompt

Few-shot examples are drawn from **outside** the 200-row sample, so no scored
row ever has its reference translation shown to the model. They are picked
deterministically from `SEED`, so every model and every re-run sees the same
examples.

Two builders: `build_completion` (flat text ending on a dangling `English:`,
for base checkpoints) and `build_chat_messages` (chat turns, for instruct
checkpoints). Cell 6 picks per model.

Sources are flattened to one line so a multi-line tweet cannot break the
`Hinglish:` / `English:` pattern a base model relies on.

In [5]:
INSTRUCTIONS = (
    "Translate Hindi-English code-mixed (Hinglish) social media messages into "
    "fluent English.\n\n"
    "The messages come from Twitter and are informal: they may contain "
    "@mentions, hashtags, emoji, and non-standard spelling. Keep @mentions and "
    "hashtags as they are. Translate the Hindi words; leave English words that "
    "are already English.\n\n"
    "Output only the English translation - no notes, no explanation, no "
    "repetition of the original."
)


def flat(text):
    return " ".join(str(text).split())


def pick_fewshot(k=3):
    used = {r["id"] for r in sample}
    pool = [r for r in all_rows
            if r["id"] not in used
            and 6 <= len(r["question"].split()) <= 22
            and usable(r)]
    random.Random(SEED + 1).shuffle(pool)
    return pool[:k]


FEWSHOT = pick_fewshot()


def build_completion(source):
    text = INSTRUCTIONS + "\n"
    for r in FEWSHOT:
        text += "\nHinglish: {}\nEnglish: {}\n".format(
            flat(r["question"]), flat(r["answer"]))
    text += "\nHinglish: {}\nEnglish:".format(flat(source))
    return text


def build_chat_messages(source):
    msgs = [{"role": "system", "content": INSTRUCTIONS}]
    for r in FEWSHOT:
        msgs.append({"role": "user", "content": flat(r["question"])})
        msgs.append({"role": "assistant", "content": flat(r["answer"])})
    msgs.append({"role": "user", "content": flat(source)})
    return msgs


print("Few-shot examples ({}), all from OUTSIDE the sample:".format(len(FEWSHOT)))
for r in FEWSHOT:
    print("  {} | {}".format(r["id"], flat(r["question"])[:64]))

print("\n" + "=" * 66)
print(build_completion(sample[0]["question"]))
print("=" * 66)
print("[reference: {}]".format(flat(sample[0]["answer"])))

Few-shot examples (3), all from OUTSIDE the sample:
  phinc_000008 | @MeThePooh jab aayega Bihar may Modi sarkaar #NitishKaNakliVikas
  phinc_000107 | Agar Ye report negative hoti to #DeMonetisation aur #GST ne desh
  phinc_001051 | .@ranaayyub tumhare liye Hindu life ka koi mayne nahi hai.

Translate Hindi-English code-mixed (Hinglish) social media messages into fluent English.

The messages come from Twitter and are informal: they may contain @mentions, hashtags, emoji, and non-standard spelling. Keep @mentions and hashtags as they are. Translate the Hindi words; leave English words that are already English.

Output only the English translation - no notes, no explanation, no repetition of the original.

Hinglish: @MeThePooh jab aayega Bihar may Modi sarkaar #NitishKaNakliVikas
English: @MeThePooh when modi government will come to bihar #NitishKaNakliVikas

Hinglish: Agar Ye report negative hoti to #DeMonetisation aur #GST ne desh ko duba Diya Par ab sab chup #easeofdoingbusiness #Ind

### Cell 6 - Generate translations and score them

Greedy decoding (`do_sample=False`) so results are reproducible, with
`max_new_tokens` scaled to the source length rather than fixed.

`clean_translation` is doing real work here. Instruction-tuned models routinely
wrap output in `Translation:` or `Here is the English:`, and base models keep
generating the next `Hinglish:` block. Left in, that boilerplate would drag
chrF++ down for a *formatting* reason and be mistaken for a bad translation. So
the cleaner strips a leading label and keeps only the first non-empty line, and
cuts anything from a following `Hinglish:` marker.

In [6]:
_LABEL = re.compile(
    r"^\s*(here(?:\s+is|'s)?\s+the\s+)?(english\s+)?translation\s*[:\-]\s*",
    re.I)
_LABEL2 = re.compile(r"^\s*english\s*[:\-]\s*", re.I)


def clean_translation(text):
    t = (text or "").strip()
    # a base model keeps going with the next few-shot block - cut it there
    t = re.split(r"\n\s*Hinglish\s*:", t)[0]
    for line in t.split("\n"):
        line = line.strip()
        if not line:
            continue
        line = _LABEL.sub("", line)
        line = _LABEL2.sub("", line)
        line = line.strip().strip('"')
        if line:
            return line
    return ""


def prompt_style(tokenizer):
    # base checkpoints have no chat template at all
    return "chat" if getattr(tokenizer, "chat_template", None) else "completion"


@torch.no_grad()
def translate(model, tokenizer, source):
    if prompt_style(tokenizer) == "completion":
        text = build_completion(source)
    else:
        msgs = build_chat_messages(source)
        try:
            text = tokenizer.apply_chat_template(
                msgs, tokenize=False, add_generation_prompt=True)
        except Exception:
            # some templates (Gemma) reject a system role - fold it into the
            # first user turn rather than dropping the instructions
            merged = [dict(m) for m in msgs[1:]]
            merged[0]["content"] = msgs[0]["content"] + "\n\n" + merged[0]["content"]
            text = tokenizer.apply_chat_template(
                merged, tokenize=False, add_generation_prompt=True)

    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    n_in   = inputs["input_ids"].shape[1]
    budget = min(4 * len(str(source).split()) + 48, MAX_NEW_TOKENS)

    out = model.generate(**inputs,
                         max_new_tokens=budget,
                         do_sample=False,
                         pad_token_id=tokenizer.pad_token_id or tokenizer.eos_token_id)
    return clean_translation(tokenizer.decode(out[0][n_in:], skip_special_tokens=True))


def clear_hf_cache():
    shutil.rmtree("/root/.cache/huggingface/hub/", ignore_errors=True)
    gc.collect()
    torch.cuda.empty_cache()


# quick check that the cleaner behaves
for raw, want in [
    ("Translation: hello there", "hello there"),
    ("Here is the English translation: hi", "hi"),
    ("English: good\nHinglish: aur kya", "good"),
    ('"quoted output"', "quoted output"),
    ("", ""),
]:
    got = clean_translation(raw)
    print("  clean {!r:<44} -> {!r}".format(raw, got))
    assert got == want, (raw, got, want)
print("\nGeneration and scoring functions defined")

  clean 'Translation: hello there'                   -> 'hello there'
  clean 'Here is the English translation: hi'        -> 'hi'
  clean 'English: good\nHinglish: aur kya'           -> 'good'
  clean '"quoted output"'                            -> 'quoted output'
  clean ''                                           -> ''

Generation and scoring functions defined


### Cell 7 - Load-or-cache, and the batched runner

`load_model` implements download-once: if `models/<name>_4bit` exists in Drive
it is loaded directly (already 4-bit, so passing a fresh `BitsAndBytesConfig`
would conflict and is omitted); otherwise the repo is downloaded, quantised,
and saved to Drive for next time. `trust_remote_code` stays off - repo-shipped
modelling code is often written against an older transformers API.

`run_model` stores the **raw chrF++ score and the translation itself**, never a
pass/fail. The threshold is applied later in Cell 9, which is what lets you
re-derive difficulty for free. Each finished batch is appended to
`judge_progress/<model>.jsonl` before the next begins, so a disconnect costs at
most `BATCH_SIZE` rows.

In [7]:
def cache_path(spec):
    return os.path.join(CACHE_DIR, spec["name"] + "_4bit")


def load_model(spec):
    cached     = cache_path(spec)
    from_drive = DRIVE_OK and os.path.isfile(os.path.join(cached, "config.json"))
    source     = cached if from_drive else spec["repo"]

    kwargs = {"device_map": "auto", "trust_remote_code": False}
    if from_drive:
        how = "Drive cache (already 4-bit)"
    else:
        kwargs["quantization_config"] = bnb_config
        how = "HuggingFace download -> 4-bit"
    if spec.get("attn"):
        kwargs["attn_implementation"] = spec["attn"]
        how += ", attn=" + spec["attn"]

    print("  loading {} [{}]".format(source, how))
    tokenizer = AutoTokenizer.from_pretrained(source)
    model = AutoModelForCausalLM.from_pretrained(source, **kwargs).eval()

    if not from_drive and DRIVE_OK:
        print("  saving 4-bit copy to {} (one time)...".format(cached))
        os.makedirs(cached, exist_ok=True)
        model.save_pretrained(cached)
        tokenizer.save_pretrained(cached)
        print("  saved - future runs skip the download")

    print("  ready | VRAM: {:.2f}GB | prompt style: {}".format(
        torch.cuda.memory_allocated() / 1e9, prompt_style(tokenizer)))
    return model, tokenizer


def run_model(spec, rows):
    prog_file = os.path.join(PROG_DIR, spec["name"] + ".jsonl")

    # ---- 1. resume from whatever is already on disk ----
    done = {}
    if os.path.exists(prog_file):
        with open(prog_file, encoding="utf-8") as f:
            for line in f:
                item = json.loads(line)
                done[item["id"]] = item
        print("  resuming - {}/{} already scored".format(len(done), len(rows)))

    remaining = [r for r in rows if r["id"] not in done]
    if not remaining:
        print("  {} already complete - skipping load".format(spec["name"]))
        return done

    # ---- 2. load (from Drive if cached, else download and cache) ----
    model, tokenizer = load_model(spec)

    # ---- 3. translate in batches, saving after each one ----
    total_batches = (len(remaining) + BATCH_SIZE - 1) // BATCH_SIZE

    for batch_start in range(0, len(remaining), BATCH_SIZE):
        batch     = remaining[batch_start : batch_start + BATCH_SIZE]
        batch_num = batch_start // BATCH_SIZE + 1

        batch_results = []
        for row in batch:
            hyp = translate(model, tokenizer, row["question"])
            batch_results.append({
                "id":      row["id"],
                "chrf":    chrf_pp(hyp, str(row["answer"])),
                "n_words": len(hyp.split()),
                "output":  hyp,
            })

        with open(prog_file, "a", encoding="utf-8") as f:
            for item in batch_results:
                f.write(json.dumps(item, ensure_ascii=False) + "\n")

        done.update({item["id"]: item for item in batch_results})
        mean_chrf = sum(v["chrf"] for v in done.values()) / len(done)
        print("  batch {}/{} saved - {}/{} rows | mean chrF++ {:.1%}".format(
            batch_num, total_batches, len(done), len(rows), mean_chrf))

    del model, tokenizer
    clear_hf_cache()
    print("  {} complete".format(spec["name"]))
    return done

print("Runner defined")

Runner defined


### Cell 8 - Run all three models

One model at a time - loaded, scored, unloaded - so peak VRAM stays near 6 GB
rather than the ~17 GB all three would need together.

This is the long cell. Every row generates a full sentence, so budget roughly
**8-12 min per model**, plus downloads on the first run. Safe to re-run:
anything already scored is skipped.

In [8]:
preds = {}
for spec in MODELS:
    print("\n=== {} ===".format(spec["name"]))
    preds[spec["name"]] = run_model(spec, sample)

print("\nAll models done")


=== mistral ===
  loading /drive/MyDrive/models/mistral_4bit [Drive cache (already 4-bit)]


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

  ready | VRAM: 4.14GB | prompt style: chat
  batch 1/12 saved - 25/300 rows | mean chrF++ 40.9%
  batch 2/12 saved - 50/300 rows | mean chrF++ 45.3%
  batch 3/12 saved - 75/300 rows | mean chrF++ 43.8%
  batch 4/12 saved - 100/300 rows | mean chrF++ 44.2%
  batch 5/12 saved - 125/300 rows | mean chrF++ 44.4%
  batch 6/12 saved - 150/300 rows | mean chrF++ 43.2%
  batch 7/12 saved - 175/300 rows | mean chrF++ 43.8%
  batch 8/12 saved - 200/300 rows | mean chrF++ 43.7%
  batch 9/12 saved - 225/300 rows | mean chrF++ 43.0%
  batch 10/12 saved - 250/300 rows | mean chrF++ 43.9%
  batch 11/12 saved - 275/300 rows | mean chrF++ 43.3%
  batch 12/12 saved - 300/300 rows | mean chrF++ 43.3%
  mistral complete

=== llama ===
  loading /drive/MyDrive/models/llama_4bit [Drive cache (already 4-bit)]


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

  ready | VRAM: 5.71GB | prompt style: chat


[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


  batch 1/12 saved - 25/300 rows | mean chrF++ 50.0%
  batch 2/12 saved - 50/300 rows | mean chrF++ 48.8%
  batch 3/12 saved - 75/300 rows | mean chrF++ 48.4%
  batch 4/12 saved - 100/300 rows | mean chrF++ 47.2%
  batch 5/12 saved - 125/300 rows | mean chrF++ 48.5%
  batch 6/12 saved - 150/300 rows | mean chrF++ 49.0%
  batch 7/12 saved - 175/300 rows | mean chrF++ 49.1%
  batch 8/12 saved - 200/300 rows | mean chrF++ 49.3%
  batch 9/12 saved - 225/300 rows | mean chrF++ 49.3%
  batch 10/12 saved - 250/300 rows | mean chrF++ 49.5%
  batch 11/12 saved - 275/300 rows | mean chrF++ 49.5%
  batch 12/12 saved - 300/300 rows | mean chrF++ 49.5%
  llama complete

=== gemma ===
  loading /drive/MyDrive/models/gemma_4bit [Drive cache (already 4-bit), attn=eager]


Loading weights:   0%|          | 0/464 [00:00<?, ?it/s]

  ready | VRAM: 6.14GB | prompt style: chat
  batch 1/12 saved - 25/300 rows | mean chrF++ 54.2%
  batch 2/12 saved - 50/300 rows | mean chrF++ 53.3%
  batch 3/12 saved - 75/300 rows | mean chrF++ 54.3%
  batch 4/12 saved - 100/300 rows | mean chrF++ 54.5%
  batch 5/12 saved - 125/300 rows | mean chrF++ 55.7%
  batch 6/12 saved - 150/300 rows | mean chrF++ 55.4%
  batch 7/12 saved - 175/300 rows | mean chrF++ 55.5%
  batch 8/12 saved - 200/300 rows | mean chrF++ 56.3%
  batch 9/12 saved - 225/300 rows | mean chrF++ 55.9%
  batch 10/12 saved - 250/300 rows | mean chrF++ 56.3%
  batch 11/12 saved - 275/300 rows | mean chrF++ 56.8%
  batch 12/12 saved - 300/300 rows | mean chrF++ 56.6%
  gemma complete

All models done


### Cell 9 - Find the threshold

The pass mark is **derived from the data**, not guessed.

It prints the score distribution per model and pooled, places the two model-free
floors from Cell 4 alongside them, and shows what every candidate threshold
would do to the Easy / Medium / Hard split. Then it picks one according to
`THRESHOLD_MODE`.

It also guards against a threshold that is technically derived but useless: one
at or below the copy-the-input floor, one so high that only a near-exact match
passes, or one that leaves a difficulty band empty. Each prints a warning
telling you to switch to `"fixed"`.

Two things to check in the output:

- The chosen threshold must sit **clearly above the copy-the-input floor**. A
  threshold below it would pass models that never translated anything.
- If one model's median is far below the others, it will fail nearly every row
  and effectively vote "Hard" throughout - contributing no signal. That is the
  degenerate case to watch for.

Nothing here re-runs a model, so you can change `THRESHOLD_MODE` in Cell 3 and
re-run just this cell and the next.

In [9]:
pooled = [preds[s["name"]][r["id"]]["chrf"] for r in sample for s in MODELS]
pooled_sorted = sorted(pooled)

def pct(p):
    return pooled_sorted[min(len(pooled_sorted) - 1, int(p * len(pooled_sorted)))]

print("chrF++ distribution per model:")
print("  {:<10} {:>7} {:>7} {:>7} {:>7}".format("model", "p25", "median", "p75", "mean"))
for s in MODELS:
    v = sorted(x["chrf"] for x in preds[s["name"]].values())
    print("  {:<10} {:>6.1%} {:>7.1%} {:>7.1%} {:>7.1%}".format(
        s["name"], v[len(v) // 4], v[len(v) // 2], v[3 * len(v) // 4],
        sum(v) / len(v)))
print("  {:<10} {:>6.1%} {:>7.1%} {:>7.1%} {:>7.1%}".format(
    "POOLED", pct(.25), pct(.50), pct(.75), sum(pooled) / len(pooled)))

print("\nmodel-free floors (from Cell 4):")
print("  copy the input : {:.1%}".format(COPY_FLOOR))
print("  unrelated text : {:.1%}".format(RAND_FLOOR))


def difficulty_at(th):
    out = Counter()
    for r in sample:
        votes = sum(preds[s["name"]][r["id"]]["chrf"] >= th for s in MODELS)
        out["Easy" if votes == 3 else ("Medium" if votes == 2 else "Hard")] += 1
    return out


# ---- pick the threshold ----
if THRESHOLD_MODE == "auto_median":
    THRESHOLD = pct(.50)
    why = "median of all pooled model scores"
elif THRESHOLD_MODE == "baseline":
    THRESHOLD = COPY_FLOOR
    why = "the copy-the-input floor"
elif THRESHOLD_MODE == "fixed":
    THRESHOLD = FIXED_THRESHOLD
    why = "FIXED_THRESHOLD from Cell 3"
else:
    raise ValueError("unknown THRESHOLD_MODE: " + str(THRESHOLD_MODE))

print("\nsensitivity - what each threshold would produce:")
print("  {:>9}  {:>6} {:>7} {:>6}".format("threshold", "Easy", "Medium", "Hard"))
cands = sorted(set([round(x, 3) for x in
                    [.30, .35, .40, .45, .50, .55, .60, .65, .70,
                     round(COPY_FLOOR, 3), round(THRESHOLD, 3)]]))
for th in cands:
    d = difficulty_at(th)
    tag = ""
    if abs(th - round(THRESHOLD, 3)) < 1e-9:
        tag += "  <- CHOSEN"
    if abs(th - round(COPY_FLOOR, 3)) < 1e-9:
        tag += "  (copy-input floor)"
    print("  {:>9.3f}  {:>6} {:>7} {:>6}{}".format(
        th, d.get("Easy", 0), d.get("Medium", 0), d.get("Hard", 0), tag))

print("\nTHRESHOLD = {:.3f}  ({})".format(THRESHOLD, why))

if THRESHOLD <= COPY_FLOOR:
    print("  WARNING: at or below the copy-the-input floor - a model could pass")
    print("  without translating anything. Set THRESHOLD_MODE='fixed'.")
else:
    print("  sits {:.1f} points above the copy-the-input floor - OK".format(
        100 * (THRESHOLD - COPY_FLOOR)))

if THRESHOLD >= 0.95:
    print("  WARNING: the threshold sits at the very top of the range. More")
    print("  than half of all scores are near-perfect, so this demands an")
    print("  almost exact match and Easy becomes unreachable. Set")
    print("  THRESHOLD_MODE='fixed' with a value from the table above.")

_bands = difficulty_at(THRESHOLD)
if min(_bands.get(k, 0) for k in ("Easy", "Medium", "Hard")) == 0:
    print("  WARNING: one difficulty band is empty at this threshold - the")
    print("  split carries little information. Pick another value.")

chrF++ distribution per model:
  model          p25  median     p75    mean
  mistral     29.9%   41.2%   55.9%   43.3%
  llama       36.4%   47.9%   62.1%   49.5%
  gemma       42.6%   56.4%   69.4%   56.6%
  POOLED      36.6%   48.1%   63.1%   49.8%

model-free floors (from Cell 4):
  copy the input : 36.7%
  unrelated text : 10.1%

sensitivity - what each threshold would produce:
  threshold    Easy  Medium   Hard
      0.300     207      58     35
      0.350     171      74     55
      0.367     162      71     67  (copy-input floor)
      0.400     133      78     89
      0.450     100      76    124
      0.481      81      73    146  <- CHOSEN
      0.500      70      67    163
      0.550      53      56    191
      0.600      34      52    214
      0.650      20      45    235
      0.700      16      26    258

THRESHOLD = 0.481  (median of all pooled model scores)
  sits 11.5 points above the copy-the-input floor - OK


### Cell 10 - Apply the threshold and write the schema

Each model votes 1 where its chrF++ clears `THRESHOLD`; the votes sum into
Easy / Medium / Hard exactly as in every other split.

Output rows are rebuilt key-by-key from `SCHEMA_KEYS`, so the file carries
exactly the 14 IndicSample fields in schema order. `difficulty` is the only
value that changes unless you set `SET_EVAL_METRIC` in Cell 3. Translations and
raw scores go to the audit file.

In [10]:
def get_difficulty(votes):
    score = sum(votes)
    if score == 3:
        return "Easy"
    elif score == 2:
        return "Medium"
    else:
        return "Hard"


final_results = []
audit = []

for row in sample:
    scores = [preds[s["name"]][row["id"]]["chrf"] for s in MODELS]
    votes  = [int(x >= THRESHOLD) for x in scores]
    difficulty = get_difficulty(votes)

    enriched = {**row, "difficulty": difficulty}
    if SET_EVAL_METRIC:
        enriched["eval_metric"] = SET_EVAL_METRIC
    final_results.append({k: enriched.get(k) for k in SCHEMA_KEYS})

    audit.append({
        "id":           row["id"],
        "difficulty":   difficulty,
        "votes":        votes,
        "chrf":         [round(x, 4) for x in scores],
        "threshold":    round(THRESHOLD, 4),
        "source":       " ".join(row["question"].split()),
        "reference":    " ".join(str(row["answer"]).split()),
        "translations": {s["name"]: preds[s["name"]][row["id"]]["output"]
                         for s in MODELS},
    })

with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    for item in final_results:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")

with open(AUDIT_FILE, "w", encoding="utf-8") as f:
    for item in audit:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")

print("Saved -> {} ({} rows)".format(OUTPUT_FILE, len(final_results)))
print("Audit -> {}".format(AUDIT_FILE))
print("threshold used: {:.3f}".format(THRESHOLD))

Saved -> phinc_difficulty.jsonl (300 rows)
Audit -> phinc_audit.jsonl
threshold used: 0.481


### Cell 11 - Verify and report

Checks before trusting the file:

1. **Schema** - all 14 keys in order, no nulls in `difficulty`.
2. **Difficulty distribution.**
3. **Per-model mean chrF++ against the copy-input floor.** A model at or below
   that floor is not translating - check its output in the audit file before
   concluding anything about difficulty.
4. **Empty-output rate.** A model returning nothing scores 0 and votes Hard for
   a formatting reason, not a linguistic one. High rates here mean the prompt or
   the cleaner needs attention, not that the rows are hard.

The sample rows at the end print the source, the reference, and all three
translations, which is the fastest way to sanity-check that Hard rows are
genuinely hard.

In [11]:
# 1. schema integrity
bad_keys = [r["id"] for r in final_results if list(r.keys()) != SCHEMA_KEYS]
missing  = [r["id"] for r in final_results if r["difficulty"] is None]
print("Schema check : {} rows | wrong keys: {} | null difficulty: {}".format(
    len(final_results), len(bad_keys), len(missing)))

# 2. difficulty distribution
dist  = Counter(r["difficulty"] for r in final_results)
total = len(final_results)
print("\nDifficulty distribution (threshold {:.3f}):".format(THRESHOLD))
for level in ["Easy", "Medium", "Hard"]:
    n = dist.get(level, 0)
    print("  {:<7}: {:4d}  ({:.1f}%)".format(level, n, n / total * 100))

# 3. per-model score vs the do-nothing floor
print("\nMean chrF++:")
for s in MODELS:
    v = [x["chrf"] for x in preds[s["name"]].values()]
    m = sum(v) / len(v)
    flag = "  <- at/below the do-nothing floor" if m <= COPY_FLOOR else ""
    print("  {:<10} {:.1%}{}".format(s["name"], m, flag))
print("  {:<10} {:.1%}  <- copy-the-input floor".format("baseline", COPY_FLOOR))

# 4. empty or truncated outputs
print("\nOutput health:")
for s in MODELS:
    v = list(preds[s["name"]].values())
    empty = sum(1 for x in v if x["n_words"] == 0)
    flag = "  <- formatting problem, not difficulty" if empty / total > 0.05 else ""
    print("  {:<10} empty {:>3}/{} | median words {}{}".format(
        s["name"], empty, total,
        sorted(x["n_words"] for x in v)[len(v) // 2], flag))

print("\n--- 2 sample rows ---")
for a in audit[:2]:
    print("\n  {} [{}] chrf={}".format(a["id"], a["difficulty"], a["chrf"]))
    print("    source   :", a["source"][:88])
    print("    reference:", a["reference"][:88])
    for k, v in a["translations"].items():
        print("    {:<8} :".format(k), v[:88])

Schema check : 300 rows | wrong keys: 0 | null difficulty: 0

Difficulty distribution (threshold 0.481):
  Easy   :   81  (27.0%)
  Medium :   73  (24.3%)
  Hard   :  146  (48.7%)

Mean chrF++:
  mistral    43.3%
  llama      49.5%
  gemma      56.6%
  baseline   36.7%  <- copy-the-input floor

Output health:
  mistral    empty   0/300 | median words 11
  llama      empty   0/300 | median words 11
  gemma      empty   0/300 | median words 12

--- 2 sample rows ---

  phinc_001335 [Hard] chrf=[0.1069, 0.1055, 0.0865]
    source   : @someUSER so they should've ran the ball on 3rd &amp
    reference: 7 ? blame cam all u want but flacco is the problem . elite qbs dont have games like that
    mistral  : @someUSER they should have run the ball on 3rd down
    llama    : @someUSER so they should have run the ball on 3rd
    gemma    : @someUSER so they should've run the ball on 3rd &

  phinc_000233 [Easy] chrf=[0.7266, 0.6177, 0.524]
    source   : @aniket51082 ye to yahi batayenge :) @bhuv